# Where to Intervene (Task 2): POMIS

Given the causal graph, POMIS prunes the intervention space to the sets that could be optimal. Engine adapted from the MIT-licensed [`sanghack81/SCMMAB-NIPS2018`](https://github.com/sanghack81/SCMMAB-NIPS2018) (Lee & Bareinboim, NeurIPS 2018).

## The POMIS of canonical graphs

In [ ]:
from causalrl import pomis
from causalrl.scm.graph import CausalGraph

bow = CausalGraph(directed_edges=[("X", "Y")], bidirected_edges=[("X", "Y")])
print("bow arc (MABUC):", pomis(bow, "Y"))  # [frozenset(), {X}]

chain = CausalGraph(
    directed_edges=[("X1", "X2"), ("X2", "X3"), ("X3", "Y")],
    bidirected_edges=[("X1", "Y")],
)
print("confounded chain:", pomis(chain, "Y"))  # [frozenset(), {X3}]

## Pruning the bandit's arms

In [ ]:
from causalrl.agents.scbandit import (
    BruteForceInterventionTS,
    FixedSetThompsonSampling,
    POMISThompsonSampling,
)
from causalrl.envs.suite.scbandit import make_confounded_chain_env

env = make_confounded_chain_env(seed=1)
print("arms:", env.action_space.n, "| optimal value:", round(env.optimal_value, 3))

## POMIS vs brute-force vs naive

In [ ]:
def tail_reward(agent, steps=8000, seed=1):
    obs, _ = env.reset(seed=seed)
    rewards = []
    for _ in range(steps):
        action = agent.act(obs)
        next_obs, reward, _, _, _ = env.step(action)
        agent.update(obs, action, reward)
        rewards.append(reward)
        obs = next_obs
    return sum(rewards[-2000:]) / 2000


print(
    "POMIS:",
    tail_reward(
        POMISThompsonSampling(env.graph, env.reward, env.arms, seed=0, manipulable=env.manipulable)
    ),
)
print("brute:", tail_reward(BruteForceInterventionTS(env.arms, seed=0), seed=2))
print("naive:", tail_reward(FixedSetThompsonSampling(env.arms, {"X3"}, seed=0), seed=3))